In [0]:
# ============================================
# CELL 1: Silver → Gold Configuration
# ============================================

storage_account = "healthcarestoragerev01"

silver_path = f"abfss://input@{storage_account}.dfs.core.windows.net/silver"
gold_path = f"abfss://input@{storage_account}.dfs.core.windows.net/gold"

print("Silver Path:", silver_path)
print("Gold Path:", gold_path)

Silver Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver
Gold Path: abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold


In [0]:
storage_account = "healthcarestoragerev01"

storage_key = dbutils.secrets.get(
    scope="healthcare-scope",
    key="storage-account-key"
)

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_key
)

print("Storage authentication configured")

Storage authentication configured


In [0]:
# ============================================
# CELL 2: Check Silver Data
# ============================================

display(
    dbutils.fs.ls(silver_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/departments/,departments/,0,1788280214000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/encounters/,encounters/,0,1788280281000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/insurance_claim_data/,insurance_claim_data/,0,1788280289000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/patients/,patients/,0,1788280296000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/providers/,providers/,0,1788280303000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/silver/transactions/,transactions/,0,1788280310000


In [0]:
# ============================================
# CELL 4: Read Silver Data
# ============================================

silver_departments = spark.read.format("delta").load(
    f"{silver_path}/departments"
)

silver_encounters = spark.read.format("delta").load(
    f"{silver_path}/encounters"
)

silver_insurance = spark.read.format("delta").load(
    f"{silver_path}/insurance_claim_data"
)

silver_patients = spark.read.format("delta").load(
    f"{silver_path}/patients"
)

silver_providers = spark.read.format("delta").load(
    f"{silver_path}/providers"
)

silver_transactions = spark.read.format("delta").load(
    f"{silver_path}/transactions"
)

print("Silver data loaded successfully")

Silver data loaded successfully


In [0]:
# ============================================
# CELL 5: Create Gold Layer
# ============================================

silver_tables = {
    "departments": silver_departments,
    "encounters": silver_encounters,
    "insurance_claim_data": silver_insurance,
    "patients": silver_patients,
    "providers": silver_providers,
    "transactions": silver_transactions
}

for table_name, df in silver_tables.items():
    output_path = f"{gold_path}/{table_name}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .save(output_path)
    )
    
    print(f"{table_name} → Gold completed")

print("Gold layer created successfully")

departments → Gold completed
encounters → Gold completed
insurance_claim_data → Gold completed
patients → Gold completed
providers → Gold completed
transactions → Gold completed
Gold layer created successfully


In [0]:
# ============================================
# CELL 6: Verify Gold Layer
# ============================================

display(
    dbutils.fs.ls(gold_path)
)

path,name,size,modificationTime
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/department_revenue/,department_revenue/,0,1788283984000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/departments/,departments/,0,1788282652000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/dim_department/,dim_department/,0,1788287104000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/dim_provider/,dim_provider/,0,1788287101000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/encounters/,encounters/,0,1788282654000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/fact_transaction/,fact_transaction/,0,1788287131000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/insurance_claim_data/,insurance_claim_data/,0,1788282657000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/patients/,patients/,0,1788282659000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/provider_revenue/,provider_revenue/,0,1788285450000
abfss://input@healthcarestoragerev01.dfs.core.windows.net/gold/providers/,providers/,0,1788282662000


In [0]:
# ==========================================
# CELL 7: Gold Data Count Verification
# ==========================================

from pyspark.sql.functions import *

tables = [
    "departments",
    "encounters",
    "insurance_claim_data",
    "patients",
    "providers",
    "transactions"
]

for table in tables:
    path = f"{gold_path}/{table}"

    df = spark.read.format("delta").load(path)

    print(f"{table} -> {df.count()} records")

print("Gold layer verification completed successfully")

departments -> 20 records
encounters -> 10000 records
insurance_claim_data -> 10000 records
patients -> 5000 records
providers -> 25 records
transactions -> 10000 records
Gold layer verification completed successfully


In [0]:
# ==========================================
# CELL 8: Verify Gold Table Schemas
# ==========================================

tables = [
    "departments",
    "encounters",
    "insurance_claim_data",
    "patients",
    "providers",
    "transactions"
]

for table in tables:
    print("\n" + "=" * 60)
    print(f"TABLE: {table}")
    print("=" * 60)

    path = f"{gold_path}/{table}"

    df = spark.read.format("delta").load(path)

    df.printSchema()

print("\nGold table schema verification completed successfully")


TABLE: departments
root
 |-- DeptID: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- _bronze_loaded_at: timestamp (nullable = true)
 |-- _silver_load_timestamp: timestamp (nullable = true)


TABLE: encounters
root
 |-- EncounterID: string (nullable = true)
 |-- PatientID: string (nullable = true)
 |-- EncounterDate: date (nullable = true)
 |-- EncounterType: string (nullable = true)
 |-- ProviderID: string (nullable = true)
 |-- DepartmentID: string (nullable = true)
 |-- ProcedureCode: integer (nullable = true)
 |-- InsertedDate: date (nullable = true)
 |-- ModifiedDate: date (nullable = true)
 |-- _bronze_loaded_at: timestamp (nullable = true)
 |-- _silver_load_timestamp: timestamp (nullable = true)


TABLE: insurance_claim_data
root
 |-- ClaimID: string (nullable = true)
 |-- TransactionID: string (nullable = true)
 |-- PatientID: string (nullable = true)
 |-- EncounterID: string (nullable = true)
 |-- ProviderID: string (nullable = true)
 |-- DeptID: string (null